# TinyLLM — Train a Tiny Language Model on Google Colab

This notebook trains a ~5M parameter GPT-style transformer on SQuAD Q&A data.

**Requirements**: Colab with GPU runtime (Runtime → Change runtime type → T4 GPU)

**Steps**:
1. Clone the repo & install dependencies
2. Train the BPE tokenizer
3. Train the model (~20-30 min on T4)
4. Interactive Q&A with the trained model

## 1. Setup: Clone repo & install dependencies

In [ ]:
# Clone the repository
!git clone https://github.com/aadimiro/tinyllm.git
%cd tinyllm

# Install sentencepiece (torch is pre-installed on Colab)
!pip install -q sentencepiece

In [ ]:
# Verify GPU is available
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB")

## 2. Train the BPE Tokenizer

Downloads SQuAD v1.1 (~30MB) and trains a Byte Pair Encoding tokenizer with 8,000 vocabulary entries.

In [ ]:
!python train_tokenizer.py

In [ ]:
# Quick test: see how the tokenizer splits text
from tokenizer import Tokenizer

tok = Tokenizer.load("data/tokenizer.model")
test = "The Eiffel Tower was built in 1889 in Paris, France."
pieces = tok.encode_as_pieces(test)
print(f"Text:   {test}")
print(f"Pieces: {pieces}")
print(f"Tokens: {len(pieces)} (compression ratio: {len(test)/len(pieces):.1f}x)")

## 3. Train the Model

This trains the 5.3M parameter transformer on SQuAD Q&A pairs.

- On a T4 GPU: ~20-30 minutes for 20K steps
- Loss should drop from ~9.0 (random) to ~2.0-2.5 (learning)
- Checkpoints are saved every 2000 steps

In [ ]:
!python train.py

### Resume training (if disconnected)

If Colab disconnects, re-run cells 1-2 above (clone + install), then:

In [ ]:
# Uncomment to resume from last checkpoint:
# !python train.py --resume checkpoints/step_10000.pt

## 4. Test the Model — Interactive Q&A

Give the model a context paragraph and a question. It will try to extract the answer from the context.

In [ ]:
from generate import load_model, generate_answer
from tokenizer import Tokenizer

# Load the best checkpoint
device = "cuda" if torch.cuda.is_available() else "cpu"
model, model_config = load_model("checkpoints/best.pt", device=device)
tokenizer = Tokenizer.load("data/tokenizer.model")

In [ ]:
# Example 1
context = """The Eiffel Tower is a wrought-iron lattice tower on the Champ de Mars
in Paris, France. It is named after the engineer Gustave Eiffel, whose company
designed and built the tower. Constructed from 1887 to 1889, it was initially
criticized by some of France's leading artists and intellectuals."""

question = "Who designed the Eiffel Tower?"

answer = generate_answer(model, tokenizer, context, question, device=device)
print(f"Q: {question}")
print(f"A: {answer}")

In [ ]:
# Example 2
context = """Python is a high-level programming language created by Guido van Rossum
and first released in 1991. Python's design philosophy emphasizes code readability
with the use of significant indentation."""

question = "When was Python first released?"

answer = generate_answer(model, tokenizer, context, question, device=device)
print(f"Q: {question}")
print(f"A: {answer}")

In [ ]:
# Try your own!
my_context = """Replace this with any paragraph of text."""
my_question = "Replace this with your question about the paragraph."

answer = generate_answer(model, tokenizer, my_context, my_question, device=device)
print(f"Q: {my_question}")
print(f"A: {answer}")

## 5. Inspect the Model

Let's look at what we built — parameter counts, memory usage, architecture.

In [ ]:
from utils import print_model_summary

print_model_summary(model)
print(f"\nContext window: {model_config.block_size} tokens")
print(f"Vocabulary: {model_config.vocab_size} tokens")
print(f"Architecture: {model_config.n_layer} layers, {model_config.n_head} heads, {model_config.n_embd} dim")

## 6. Save model to Google Drive (optional)

Save the trained model to your Drive so you don't lose it if Colab disconnects.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!mkdir -p /content/drive/MyDrive/tinyllm_checkpoints
!cp checkpoints/best.pt /content/drive/MyDrive/tinyllm_checkpoints/
!cp data/tokenizer.model /content/drive/MyDrive/tinyllm_checkpoints/
print("Model saved to Google Drive!")